# EEG Motor Imagery — Parallel CNN-GRU
**Vertex AI Workbench** · Subject-Independent 3-Way Split

Architecture: parallel CNN + GRU branches fused before the classification head.
- **Subjects 1-10** → Train / Validation pool  
- **Subjects 11-12** → Blind test set (never seen during training)

Reference: Cui et al., *EEGNET + Grad-CAM for EEG Motor Imagery Decoding*, 2023.

## 1. Install dependencies

In [ ]:
# Run once; kernel restart not required on Vertex AI Workbench (packages install into the active env)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q numpy pandas matplotlib seaborn scikit-learn tqdm google-cloud-storage

## 2. Imports & device

In [ ]:
import os
import sys
import json
import time
from collections import defaultdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix

# ── Device ────────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    DEVICE = torch.device("cpu")
    print("No GPU found — running on CPU")

print(f"Device: {DEVICE}")

## 3. Configuration
Edit the paths and hyperparameters here. Everything downstream reads from this cell.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Set GCS_BUCKET to your bucket name (e.g. "my-eeg-bucket").
# Leave DATA_DIR / OUTPUT_DIR as local paths — the download cell below
# will pull data from GCS into these locations.
GCS_BUCKET      = "parallel_eeg_decoding"          # ← change me
GCS_DATA_PREFIX = "eeg_data/npz_4class_parallel_W10"        # path inside the bucket

DATA_DIR   = "/home/jupyter/eeg_data/npz_4class_parallel_W10"
OUTPUT_DIR = "eeg_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Dataset ───────────────────────────────────────────────────────────────────
WINDOW       = 10
N_CLASSES    = 4
SEED         = 42
TRAIN_RATIO  = 0.8    # fraction of subjects 1-10 used for training
NUM_SUBJECTS = 12     # 10 train/val + 2 blind test

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS  = 10
BATCH   = 64          # increase if GPU has ≥ 16 GB VRAM
LR      = 1e-4
DROPOUT = 0.5

GRAD_ACCUM_STEPS = 2  # effective batch = BATCH * GRAD_ACCUM_STEPS
LR_T_MAX         = EPOCHS
LR_ETA_MIN       = 1e-6
NOISE_STD        = 0.05   # Gaussian noise std for training augmentation; 0.0 = off

CHECKPOINT_INTERVAL    = 10
EARLY_STOPPING_PATIENCE = 15
SWEEP_EPOCHS           = 10

# ── DataLoader ────────────────────────────────────────────────────────────────
NUM_WORKERS     = 4          # safe to use > 0 on Vertex AI (no MPS fork issue)
PREFETCH_FACTOR = 2
PIN_MEMORY      = DEVICE.type == "cuda"

# ── Model ─────────────────────────────────────────────────────────────────────
PARALLEL_CFG = dict(
    conv_channels = 32,
    cnn_fc        = 256,
    n_electrodes  = 64,
    rnn_fc_in     = 256,
    gru_hidden    = 16,
    gru_layers    = 2,
    rnn_fc_out    = 256,
    fusion        = "add",
)

print(f"Effective batch size: {BATCH * GRAD_ACCUM_STEPS}")
print(f"Output dir: {OUTPUT_DIR}")

## 4. Download data from GCS
Skip this cell if the data is already on local disk.

In [ ]:
from google.cloud import storage

def download_gcs_prefix(bucket_name: str, prefix: str, local_dir: str) -> None:
    """Download all blobs under `prefix` in `bucket_name` to `local_dir`."""
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    blobs  = list(client.list_blobs(bucket_name, prefix=prefix))
    os.makedirs(local_dir, exist_ok=True)

    for blob in tqdm(blobs, desc="Downloading"):
        rel_path   = os.path.relpath(blob.name, prefix)
        local_path = os.path.join(local_dir, rel_path)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        if not os.path.exists(local_path):
            blob.download_to_filename(local_path)

    print(f"Downloaded {len(blobs)} file(s) to {local_dir}")

# Uncomment to run:
# download_gcs_prefix(GCS_BUCKET, GCS_DATA_PREFIX, DATA_DIR)

## 5. Dataset

In [ ]:
class NoisySubset(Dataset):
    """Wraps a Subset and injects Gaussian noise into EEG tensors during training."""

    def __init__(self, subset: Dataset, noise_std: float = 0.0):
        self.subset    = subset
        self.noise_std = noise_std

    def __len__(self) -> int:
        return len(self.subset)

    def __getitem__(self, idx: int):
        cnn_x, rnn_x, label = self.subset[idx]
        if self.noise_std > 0.0:
            cnn_x = cnn_x + torch.randn_like(cnn_x) * self.noise_std
            rnn_x = rnn_x + torch.randn_like(rnn_x) * self.noise_std
        return cnn_x, rnn_x, label


class EEGDataset(Dataset):
    """Memory-mapped .npz dataset for the Parallel CNN-GRU model."""

    def __init__(self, data_dir: str, num_subjects: int = 12):
        labels_path = os.path.join(data_dir, "S001_S108_win10_labels.npz")
        cnn_path    = os.path.join(data_dir, "S001_S108_win10_cnn_data.npz")
        rnn_path    = os.path.join(data_dir, "S001_S108_win10_rnn_data.npz")

        if not all(os.path.exists(p) for p in [labels_path, cnn_path, rnn_path]):
            raise FileNotFoundError(f"Missing .npz files in {data_dir}")

        print("[EEGDataset] Loading labels …")
        labels_raw   = np.load(labels_path, allow_pickle=True)["labels"]
        self.classes = sorted(list(set(labels_raw)))
        c2i          = {c: i for i, c in enumerate(self.classes)}
        self.labels  = np.array([c2i[l] for l in labels_raw], dtype=np.int64)

        print("[EEGDataset] Loading CNN data …")
        self.cnn_data = np.load(cnn_path)["data"]

        print("[EEGDataset] Loading RNN data …")
        self.rnn_data = np.load(rnn_path)["data"]

        self.total_len           = len(self.labels)
        self.samples_per_subject = self.total_len / 108

        if num_subjects and 0 < num_subjects < 108:
            self.limit_idx = int(num_subjects * self.samples_per_subject)
            print(f"[EEGDataset] Limited to first {num_subjects} subjects ({self.limit_idx:,} samples)")
        else:
            self.limit_idx = self.total_len

    def __len__(self) -> int:
        return self.limit_idx

    def __getitem__(self, idx: int):
        cnn   = self.cnn_data[idx].astype(np.float32)[:, np.newaxis, :, :]
        rnn   = self.rnn_data[idx].astype(np.float32)
        label = int(self.labels[idx])
        return torch.from_numpy(cnn), torch.from_numpy(rnn), label


def build_loaders(
    data_dir:     str   = DATA_DIR,
    num_subjects: int   = NUM_SUBJECTS,
    train_ratio:  float = TRAIN_RATIO,
    batch_size:   int   = BATCH,
    num_workers:  int   = NUM_WORKERS,
    pin_memory:   bool  = PIN_MEMORY,
    seed:         int   = SEED,
) -> Tuple[DataLoader, DataLoader, DataLoader, list]:
    """Build Train / Val / Blind Test loaders with subject-independent split."""
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)

    full_dataset = EEGDataset(data_dir, num_subjects=num_subjects)
    total_len    = len(full_dataset)

    test_subject_count     = 2
    train_val_subject_count = num_subjects - test_subject_count
    split_idx = int(train_val_subject_count * full_dataset.samples_per_subject)

    test_set = Subset(full_dataset, list(range(split_idx, total_len)))
    pool_set = Subset(full_dataset, list(range(0, split_idx)))

    n_train = int(train_ratio * len(pool_set))
    n_val   = len(pool_set) - n_train
    train_set, val_set = random_split(pool_set, [n_train, n_val], generator=generator)

    noisy_train_set = NoisySubset(train_set, noise_std=NOISE_STD)

    _loader_kwargs = dict(
        batch_size        = batch_size,
        num_workers       = num_workers,
        pin_memory        = pin_memory,
        persistent_workers = (num_workers > 0),
        **(dict(prefetch_factor=PREFETCH_FACTOR) if num_workers > 0 else {}),
    )

    train_loader = DataLoader(noisy_train_set, shuffle=True,  **_loader_kwargs)
    val_loader   = DataLoader(val_set,         shuffle=False, **_loader_kwargs)
    test_loader  = DataLoader(test_set,        shuffle=False, **_loader_kwargs)

    noise_tag = f"std={NOISE_STD}" if NOISE_STD > 0 else "disabled"
    print(f"\n[build_loaders] Subject-Independent Split:")
    print(f"  Subjects 01-10  → Train: {len(train_set):,}  |  Val: {len(val_set):,}")
    print(f"  Subjects 11-12  → Test : {len(test_set):,}")
    print(f"  Total active    : {total_len:,}")
    print(f"  Gaussian noise  : {noise_tag}")

    return train_loader, val_loader, test_loader, full_dataset.classes

## 6. Model

In [ ]:
class ParallelCNNGRU(nn.Module):
    """
    Parallel CNN + GRU with configurable fusion.

    Data flow:
        CNN branch : (B,W,1,H,Wd) → 2-D CNN per frame → sum over W → cnn_fc vector
        GRU branch : (B,W,n_electrodes) → linear projection → GRU → rnn_fc vector
        Fusion     : 'concat' | 'add' | 'concat_fc' | 'concat_conv1d'
        Output     : (B, n_classes) raw logits
    """

    def __init__(
        self,
        window_size:   int   = 10,
        conv_channels: int   = 32,
        cnn_fc:        int   = 256,
        n_electrodes:  int   = 64,
        rnn_fc_in:     int   = 256,
        gru_hidden:    int   = 16,
        gru_layers:    int   = 2,
        rnn_fc_out:    int   = 256,
        n_classes:     int   = 4,
        dropout:       float = 0.5,
        fusion:        str   = "add",
        **kwargs,
    ):
        super().__init__()
        self.fusion = fusion
        ch = conv_channels

        # ── CNN branch ─────────────────────────────────────────────────────────
        self.cnn_features = nn.Sequential(
            nn.Conv2d(1,    ch,   3, padding=1), nn.BatchNorm2d(ch),   nn.ELU(inplace=True),
            nn.Conv2d(ch,   ch*2, 3, padding=1), nn.BatchNorm2d(ch*2), nn.ELU(inplace=True),
            nn.Conv2d(ch*2, ch*4, 3, padding=1), nn.BatchNorm2d(ch*4), nn.ELU(inplace=True),
            nn.Conv2d(ch*4, ch*8, 3, padding=1), nn.BatchNorm2d(ch*8), nn.ELU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.cnn_fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(ch * 8, cnn_fc),
            nn.ELU(inplace=True),
            nn.Dropout(dropout),
        )

        self.bidirectional = kwargs.get("bidirectional", False)
        self.rnn_proj = nn.Sequential(
            nn.Linear(n_electrodes, rnn_fc_in),
            nn.ELU(inplace=True),
        )
        self.gru = nn.GRU(
            rnn_fc_in, gru_hidden, gru_layers,
            batch_first  = True,
            dropout      = dropout if gru_layers > 1 else 0.0,
            bidirectional = self.bidirectional,
        )
        rnn_in_size = gru_hidden * 2 if self.bidirectional else gru_hidden
        self.rnn_head = nn.Sequential(
            nn.Linear(rnn_in_size, rnn_fc_out),
            nn.ELU(inplace=True),
            nn.Dropout(dropout),
        )

        # ── Fusion layer ────────────────────────────────────────────────────────
        if fusion == "concat":
            fused_dim       = cnn_fc + rnn_fc_out
            self.fuse_layer = None
        elif fusion == "add":
            if cnn_fc != rnn_fc_out:
                raise ValueError(f"'add' fusion requires cnn_fc == rnn_fc_out ({cnn_fc} vs {rnn_fc_out})")
            fused_dim       = cnn_fc
            self.fuse_layer = None
        elif fusion == "concat_fc":
            fused_dim       = cnn_fc + rnn_fc_out
            self.fuse_layer = nn.Sequential(nn.Linear(fused_dim, fused_dim), nn.ELU(inplace=True))
        elif fusion == "concat_conv1d":
            fused_dim       = cnn_fc + rnn_fc_out
            self.fuse_layer = nn.Sequential(nn.Conv1d(fused_dim, fused_dim, kernel_size=1), nn.ELU(inplace=True))
        else:
            raise ValueError(f"Unknown fusion strategy: {fusion!r}")

        self.readout = nn.Linear(fused_dim, n_classes)

    def forward(self, cnn_x: torch.Tensor, rnn_x: torch.Tensor) -> torch.Tensor:
        B, W, C, H, Wd = cnn_x.shape

        cnn_frames = self.cnn_fc(self.cnn_features(cnn_x.reshape(B * W, C, H, Wd)))
        cnn_out    = cnn_frames.reshape(B, W, -1).sum(dim=1)

        proj    = self.rnn_proj(rnn_x.reshape(B * W, -1)).reshape(B, W, -1)
        gru_out, _ = self.gru(proj)
        rnn_out = self.rnn_head(gru_out[:, -1, :])

        if self.fusion == "concat":
            fused = torch.cat([cnn_out, rnn_out], dim=1)
        elif self.fusion == "add":
            fused = cnn_out + rnn_out
        elif self.fusion == "concat_fc":
            fused = self.fuse_layer(torch.cat([cnn_out, rnn_out], dim=1))
        elif self.fusion == "concat_conv1d":
            cat   = torch.cat([cnn_out, rnn_out], dim=1).unsqueeze(2)
            fused = self.fuse_layer(cat).squeeze(2)

        return self.readout(fused)


def build_model(cfg: dict, window: int, n_classes: int, dropout: float) -> ParallelCNNGRU:
    return ParallelCNNGRU(window_size=window, n_classes=n_classes, dropout=dropout, **cfg)


# Quick sanity check
with torch.no_grad():
    _m = build_model(PARALLEL_CFG, WINDOW, N_CLASSES, DROPOUT)
    _B, _W, _H, _Wd = 2, WINDOW, 10, 64
    _cnn = torch.zeros(_B, _W, 1, _H, _Wd)
    _rnn = torch.zeros(_B, _W, 64)
    _out = _m(_cnn, _rnn)
    n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
    print(f"Output shape : {tuple(_out.shape)}  (expect ({_B}, {N_CLASSES}))")
    print(f"Parameters   : {n_params:,}")
    del _m, _cnn, _rnn, _out

## 7. Trainer

In [ ]:
History = Dict[str, List[float]]


def train_epoch(
    model:      nn.Module,
    loader:     DataLoader,
    criterion:  nn.Module,
    optimiser:  torch.optim.Optimizer,
    scaler:     GradScaler,
    device:     torch.device,
    accum_steps: int = 1,
) -> Tuple[float, float]:
    model.train()
    total_loss, correct, n_samples = 0.0, 0, 0
    bar = tqdm(loader, desc="  train", leave=False)

    optimiser.zero_grad(set_to_none=True)

    for step, (cnn_x, rnn_x, labels) in enumerate(bar):
        cnn_x  = cnn_x.to(device, dtype=torch.float32, non_blocking=True)
        rnn_x  = rnn_x.to(device, dtype=torch.float32, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with autocast(device_type=device.type):
            logits = model(cnn_x, rnn_x)
            loss   = criterion(logits, labels) / accum_steps

        scaler.scale(loss).backward()

        total_loss += loss.item() * accum_steps * labels.size(0)
        correct    += (logits.argmax(dim=1) == labels).sum().item()
        n_samples  += labels.size(0)

        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            scaler.step(optimiser)
            scaler.update()
            optimiser.zero_grad(set_to_none=True)

        bar.set_postfix(loss=f"{total_loss/n_samples:.4f}", acc=f"{correct/n_samples:.4f}")

    return total_loss / n_samples, correct / n_samples


@torch.no_grad()
def evaluate(
    model:     nn.Module,
    loader:    DataLoader,
    criterion: nn.Module,
    device:    torch.device,
) -> Tuple[float, float, List[int], List[int]]:
    model.eval()
    total_loss, correct, n_samples = 0.0, 0, 0
    all_preds, all_labels = [], []

    for cnn_x, rnn_x, labels in tqdm(loader, desc="   eval", leave=False):
        cnn_x  = cnn_x.to(device, dtype=torch.float32, non_blocking=True)
        rnn_x  = rnn_x.to(device, dtype=torch.float32, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(cnn_x, rnn_x)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        n_samples  += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / n_samples, correct / n_samples, all_preds, all_labels


def _save_checkpoint(path, epoch, model, optimiser, scaler, scheduler, best_val_acc, history):
    torch.save({
        "epoch":             epoch,
        "model_state_dict":  model.state_dict(),
        "optim_state_dict":  optimiser.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "sched_state_dict":  scheduler.state_dict(),
        "best_val_acc":      best_val_acc,
        "history":           dict(history),
    }, path)


def run_training(
    model:       nn.Module,
    train_loader: DataLoader,
    val_loader:  DataLoader,
    device:      torch.device = DEVICE,
    epochs:      int          = EPOCHS,
    lr:          float        = LR,
    output_dir:  str          = OUTPUT_DIR,
    accum_steps: int          = GRAD_ACCUM_STEPS,
    early_stopping_patience: int = EARLY_STOPPING_PATIENCE,
) -> Tuple[History, float]:
    criterion = nn.CrossEntropyLoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler    = GradScaler(enabled=(device.type == "cuda"))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimiser, T_max=LR_T_MAX, eta_min=LR_ETA_MIN
    )

    ckpt_path   = os.path.join(output_dir, "best_parallel.pt")
    resume_path = os.path.join(output_dir, "parallel_resume.pt")

    best_val_acc    = 0.0
    history         = defaultdict(list)
    start_epoch     = 1
    patience_counter = 0

    if os.path.exists(resume_path):
        print(f"[trainer] Resuming from {resume_path} …")
        ckpt = torch.load(resume_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimiser.load_state_dict(ckpt["optim_state_dict"])
        if "scaler_state_dict" in ckpt:   scaler.load_state_dict(ckpt["scaler_state_dict"])
        if "sched_state_dict" in ckpt:    scheduler.load_state_dict(ckpt["sched_state_dict"])
        start_epoch  = ckpt["epoch"] + 1
        best_val_acc = ckpt.get("best_val_acc", 0.0)
        history      = defaultdict(list, ckpt["history"])

    model.to(device)
    eff_batch = train_loader.batch_size * accum_steps
    print(f"[trainer] epochs {start_epoch}→{epochs} | device={device} | eff_batch={eff_batch}")

    epoch_bar = tqdm(range(start_epoch, epochs + 1), desc="Epochs", initial=start_epoch - 1, total=epochs)

    try:
        for epoch in epoch_bar:
            t0 = time.perf_counter()
            tr_loss, tr_acc        = train_epoch(model, train_loader, criterion, optimiser, scaler, device, accum_steps)
            vl_loss, vl_acc, _, _  = evaluate(model, val_loader, criterion, device)

            scheduler.step()
            current_lr = scheduler.get_last_lr()[0]

            history["tr_loss"].append(tr_loss);  history["tr_acc"].append(tr_acc)
            history["vl_loss"].append(vl_loss);  history["vl_acc"].append(vl_acc)
            history["lr"].append(current_lr)

            new_best = ""
            if vl_acc > best_val_acc:
                best_val_acc     = vl_acc
                patience_counter = 0
                torch.save(model.state_dict(), ckpt_path)
                new_best = " *"
            else:
                patience_counter += 1

            if epoch % CHECKPOINT_INTERVAL == 0 or epoch == epochs:
                snapshot = os.path.join(output_dir, f"parallel_epoch_{epoch}.pt")
                _save_checkpoint(snapshot, epoch, model, optimiser, scaler, scheduler, best_val_acc, history)
                _save_checkpoint(resume_path, epoch, model, optimiser, scaler, scheduler, best_val_acc, history)
                pd.DataFrame(dict(history)).to_csv(os.path.join(output_dir, "parallel_history.csv"), index_label="epoch_idx")

            if DEVICE.type == "cuda": torch.cuda.empty_cache()

            epoch_bar.set_postfix(
                tr_acc = f"{tr_acc:.4f}",
                vl_acc = f"{vl_acc:.4f}",
                best   = f"{best_val_acc:.4f}",
                lr     = f"{current_lr:.2e}",
                s      = f"{time.perf_counter()-t0:.1f}",
            )
            if new_best:
                tqdm.write(f"  Ep {epoch:3d}/{epochs}{new_best}  lr={current_lr:.2e}  val_acc={vl_acc:.4f}")

            if patience_counter >= early_stopping_patience:
                tqdm.write(f"Early stopping at epoch {epoch} (no improvement in {early_stopping_patience} epochs)")
                break
    except KeyboardInterrupt:
        tqdm.write("Training interrupted.")

    return dict(history), best_val_acc

## 8. Evaluation & Plotting

In [ ]:
def plot_training_curves(history: History, best_acc: float, output_dir: str = OUTPUT_DIR) -> None:
    if not history or not history.get("tr_loss"):
        return
    ep = range(1, len(history["tr_loss"]) + 1)

    fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Parallel CNN-GRU — Training Curves", fontweight="bold")

    ax_loss.plot(ep, history["tr_loss"], label="Train", color="steelblue", lw=1.5)
    ax_loss.plot(ep, history["vl_loss"], label="Val",   color="orangered",  lw=1.5)
    ax_loss.set(xlabel="Epoch", ylabel="Loss", title="Cross-Entropy Loss")
    ax_loss.legend(); ax_loss.grid(alpha=0.3)

    ax_acc.plot(ep, [a * 100 for a in history["tr_acc"]], label="Train", color="steelblue", lw=1.5)
    ax_acc.plot(ep, [a * 100 for a in history["vl_acc"]], label="Val",   color="orangered",  lw=1.5)
    ax_acc.axhline(best_acc * 100, ls="--", color="green", alpha=0.7, label=f"Best Val {best_acc*100:.2f}%")
    ax_acc.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax_acc.set(xlabel="Epoch", ylabel="Accuracy (%)", title="Accuracy")
    ax_acc.legend(); ax_acc.grid(alpha=0.3)

    plt.tight_layout()
    out_path = os.path.join(output_dir, "parallel_curves.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")


def plot_confusion_matrix(true_labels, predictions, class_names, output_dir=OUTPUT_DIR) -> np.ndarray:
    cm      = confusion_matrix(true_labels, predictions)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Independent Test Set (Subjects 11-12) — Confusion Matrix", fontweight="bold")

    for ax, data, fmt, title in zip(axes, [cm, cm_norm], ["d", ".2f"], ["Counts", "Normalised"]):
        sns.heatmap(data, ax=ax, annot=True, fmt=fmt, cmap="Blues",
                    xticklabels=class_names, yticklabels=class_names, linewidths=0.4)
        ax.set(xlabel="Predicted", ylabel="True", title=title)
        ax.tick_params(axis="x", rotation=30)

    plt.tight_layout()
    out_path = os.path.join(output_dir, "parallel_confusion.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")
    return cm_norm


def plot_per_class_accuracy(cm_norm, class_names, overall_acc, output_dir=OUTPUT_DIR) -> None:
    per_class = cm_norm.diagonal()
    colors    = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"] + ["#8172B2"] * 10

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(class_names, per_class * 100, color=colors[:len(class_names)], edgecolor="white", linewidth=0.8)
    for bar, v in zip(bars, per_class):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{v*100:.1f}%", ha="center", fontsize=10, fontweight="bold")
    ax.axhline(overall_acc * 100, ls="--", color="black", alpha=0.5, label=f"Overall {overall_acc*100:.1f}%")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
    ax.set(ylabel="Accuracy (%)", title="Per-Class Test Accuracy", ylim=(0, 115))
    ax.legend(); ax.grid(axis="y", alpha=0.3); ax.tick_params(axis="x", rotation=15)
    plt.tight_layout()

    out_path = os.path.join(output_dir, "parallel_per_class.png")
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")


def run_final_evaluation(
    model:       nn.Module,
    test_loader: DataLoader,
    class_names: list,
    history:     History,
    best_acc:    float,
    device:      torch.device = DEVICE,
    output_dir:  str          = OUTPUT_DIR,
    ckpt_path:   str | None   = None,
) -> None:
    if ckpt_path is None:
        ckpt_path = os.path.join(output_dir, "best_parallel.pt")

    print(f"[evaluate] Loading best weights from {ckpt_path}")
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    _, final_acc, preds, true_labels = evaluate(model, test_loader, criterion, device)

    print(f"\nFINAL INDEPENDENT TEST ACCURACY: {final_acc:.4f}  ({final_acc*100:.2f}%)")
    print("\n── Classification Report ──")
    print(classification_report(true_labels, preds, target_names=class_names))

    plot_training_curves(history, best_acc, output_dir)
    cm_norm = plot_confusion_matrix(true_labels, preds, class_names, output_dir)
    plot_per_class_accuracy(cm_norm, class_names, final_acc, output_dir)

## 9. Fusion hyperparameter sweep (optional)

In [ ]:
def _run_single_sweep(cfg_overrides, train_loader, val_loader, device, sweep_epochs, lr):
    merged_cfg = {**PARALLEL_CFG, **cfg_overrides}
    model      = build_model(merged_cfg, WINDOW, N_CLASSES, DROPOUT).to(device)
    criterion  = nn.CrossEntropyLoss()
    optimiser  = torch.optim.Adam(model.parameters(), lr=lr)
    scaler     = GradScaler(enabled=(device.type == "cuda"))

    vl_accs, best_vl_acc, best_weights = [], 0.0, None

    for _ in tqdm(range(sweep_epochs), desc="  epochs", leave=False):
        train_epoch(model, train_loader, criterion, optimiser, scaler, device)
        _, vl_acc, _, _ = evaluate(model, val_loader, criterion, device)
        vl_accs.append(vl_acc)
        if vl_acc > best_vl_acc:
            best_vl_acc  = vl_acc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    del model, optimiser
    if device.type == "cuda": torch.cuda.empty_cache()
    return vl_accs, best_weights


def run_sweep(
    train_loader: DataLoader,
    val_loader:   DataLoader,
    device:       torch.device = DEVICE,
    sweep_epochs: int          = SWEEP_EPOCHS,
    lr:           float        = LR,
    output_dir:   str          = OUTPUT_DIR,
) -> None:
    sweep_configs = [
        {"name": "concat",        "fusion": "concat"},
        {"name": "concat_fc",     "fusion": "concat_fc"},
        {"name": "concat_conv1d", "fusion": "concat_conv1d"},
        {"name": "add (256)",     "fusion": "add", "cnn_fc": 256, "rnn_fc_in": 256, "rnn_fc_out": 256},
    ]

    results = []
    for raw_cfg in tqdm(sweep_configs, desc="Sweep configs"):
        name      = raw_cfg["name"]
        overrides = {k: v for k, v in raw_cfg.items() if k != "name"}
        vl_accs, best_weights = _run_single_sweep(overrides, train_loader, val_loader, device, sweep_epochs, lr)
        best = max(vl_accs)
        tqdm.write(f"  {name:25s}  best_val_acc={best:.4f}")
        results.append({"name": name, "best_acc": best, "history": vl_accs})

        fn_safe = name.replace(" ", "_").replace("(", "").replace(")", "")
        torch.save(best_weights, os.path.join(output_dir, f"best_sweep_{fn_safe}.pt"))
        with open(os.path.join(output_dir, "parallel_sweep_results.json"), "w") as f:
            json.dump(results, f, indent=4)

        # Live plot
        fig, ax = plt.subplots(figsize=(10, 5))
        for r in results:
            ax.plot(range(1, sweep_epochs + 1), [a * 100 for a in r["history"]], lw=2,
                    label=f"{r['name']}  (best {r['best_acc']*100:.2f}%)")
        ax.set(xlabel="Epoch", ylabel="Val Acc (%)", title="Fusion Sweep")
        ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
        plt.savefig(os.path.join(output_dir, "parallel_sweep.png"), dpi=150, bbox_inches="tight")
        plt.show()

    print("\nRanking:")
    for r in sorted(results, key=lambda x: x["best_acc"], reverse=True):
        print(f"  {r['name']:25s}  {r['best_acc']:.4f}")

## 10. Run
### 10a. Build DataLoaders

In [ ]:
train_loader, val_loader, test_loader, class_names = build_loaders()
print(f"Classes: {class_names}")

### 10b. Build model

In [ ]:
model = build_model(
    cfg       = PARALLEL_CFG,
    window    = WINDOW,
    n_classes = N_CLASSES,
    dropout   = DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_params:,}")

### 10c. Train

In [ ]:
history, best_val_acc = run_training(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = DEVICE,
    epochs       = EPOCHS,
    lr           = LR,
    output_dir   = OUTPUT_DIR,
)
print(f"Best validation accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)")

### 10d. Final blind-test evaluation

In [ ]:
run_final_evaluation(
    model        = model,
    test_loader  = test_loader,
    class_names  = class_names,
    history      = history if history.get("tr_loss") else None,
    best_acc     = best_val_acc,
    device       = DEVICE,
    output_dir   = OUTPUT_DIR,
)

### 10e. (Optional) Fusion sweep

In [ ]:
# Uncomment to run the fusion strategy sweep
# run_sweep(
#     train_loader = train_loader,
#     val_loader   = val_loader,
#     device       = DEVICE,
#     output_dir   = OUTPUT_DIR,
# )

## 11. (Optional) Upload results to GCS

In [ ]:
def upload_dir_to_gcs(local_dir: str, bucket_name: str, gcs_prefix: str) -> None:
    """Upload every file in `local_dir` to gs://<bucket_name>/<gcs_prefix>/."""
    from google.cloud import storage
    client = storage.Client()
    bucket = client.bucket(bucket_name)
    files  = [f for f in os.listdir(local_dir) if os.path.isfile(os.path.join(local_dir, f))]
    for fname in tqdm(files, desc="Uploading"):
        blob = bucket.blob(f"{gcs_prefix}/{fname}")
        blob.upload_from_filename(os.path.join(local_dir, fname))
    print(f"Uploaded {len(files)} file(s) to gs://{bucket_name}/{gcs_prefix}/")

# Uncomment to upload:
# upload_dir_to_gcs(OUTPUT_DIR, GCS_BUCKET, "results")